In [38]:
pip install sentence-transformers scikit-learn gradio numpy

In [39]:
import numpy as np
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import CountVectorizer

In [40]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [41]:
def calculate_similarity(resume_text, job_text):

    resume_embedding = model.encode(resume_text, convert_to_tensor=True)
    job_embedding = model.encode(job_text, convert_to_tensor=True)

    similarity = util.cos_sim(resume_embedding, job_embedding)

    score = float(similarity[0][0])

    return round(score * 100, 2)

In [43]:
def extract_skills(text):

    vectorizer = CountVectorizer(
        stop_words="english",
        ngram_range=(1,3),
        max_features=50
    )

    vectorizer.fit([text])

    return set(vectorizer.get_feature_names_out())

In [44]:
def compare_skills(resume_text, job_text):

    resume_skills = extract_skills(resume_text)
    job_skills = extract_skills(job_text)

    matched = resume_skills.intersection(job_skills)
    missing = job_skills.difference(resume_skills)

    return list(matched), list(missing)

In [45]:
def analyze_resume(resume_text, job_text):

    # Similarity score
    score = calculate_similarity(resume_text, job_text)

    # Skill comparison
    matched, missing = compare_skills(resume_text, job_text)

    result = {
        "Match Score (%)": score,
        "Matched Skills": matched[:10],
        "Missing Skills": missing[:10]
    }

    return result

In [46]:
resume = """
Python developer with experience in AWS, SQL, machine learning,
data analysis, and NLP.
"""

job_description = """
Looking for a data scientist with Python, machine learning,
deep learning, SQL, and cloud computing experience.
"""

print(analyze_resume(resume, job_description))

{'Match Score (%)': 67.55, 'Matched Skills': ['learning', 'python', 'experience', 'machine', 'machine learning', 'data', 'sql'], 'Missing Skills': ['scientist python', 'learning deep', 'learning sql', 'data scientist python', 'cloud computing experience', 'sql cloud computing', 'deep learning sql', 'learning sql cloud', 'deep learning', 'scientist']}


In [47]:
import gradio as gr

def gradio_analyze(resume_text, job_text):
    result = analyze_resume(resume_text, job_text)

    output = f"""
    Match Score: {result['Match Score (%)']}%

    Matched Skills:
    {result['Matched Skills']}

    Missing Skills:
    {result['Missing Skills']}
    """

    return output


interface = gr.Interface(
    fn=gradio_analyze,
    inputs=[
        gr.Textbox(label="Paste Resume Text"),
        gr.Textbox(label="Paste Job Description")
    ],
    outputs="text",
    title="AI Resume Analyzer"
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8f0dbc96fdd3101b1a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
